Aqui vamos a desormalizar las columnas del parquet entrenado por el GAN.

1. Product_Cateogory (DONE!!)
2. Zonas dropoff_zone_id (DONE!!)

In [3]:
# ==============================================================================
# CELL 3: INICIALIZACIÓN DE SPARK SESSION (Sovereign Mode)
# ==============================================================================
import os
import findspark
from pyspark.sql import SparkSession
import warnings

# --- 1. RUTA DE SOBERANÍA (Credenciales) ---
SA_PATH = "/workspaces/pienza/secrets/service-account.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SA_PATH

# --- 2. LOCALIZAR SPARK ---
findspark.init()

# --- 3. CONSTRUCCIÓN DEL MOTOR ---
# Nota: Usamos 4g de RAM para manejar el Manifold del millón con soltura
spark = SparkSession.builder \
    .appName("Pienza_Denormalization") \
    .config("spark.jars.packages", "com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.34.0") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# --- 4. CONFIGURACIÓN HADOOP (Para puente GCS-BigQuery) ---
# Esto permite que Spark use tu Service Account para mover datos a la nube
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.enable", "true")
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile", SA_PATH)

print(f"--- Spark Iniciado en Modo Soberano ---")
print(f"Versión: {spark.version}")
print(f"🔗 Conector BigQuery cargado y autenticado via SA.")

--- Spark Iniciado en Modo Soberano ---
Versión: 3.5.1
🔗 Conector BigQuery cargado y autenticado via SA.


In [4]:
# ==============================================================================
# CELL 4: CARGA DE DATOS LOCALES (MANIFOLD V8)
# ==============================================================================
# Purpose: Load the synthetic manifold from the local workspace into Spark.
# ==============================================================================

# 1. Ruta exacta al dataset en tu Codespace (Output del GAN V8)
# Ajustado a la estructura de carpetas de tu repo local
PARQUET_SOURCE_PATH = "/workspaces/pienza/data/dumped_files/260426_cGAN_manifold_v8.parquet"

print(f"📂 Cargando dataset maestro desde: {PARQUET_SOURCE_PATH} ...")

# 2. Leer el archivo Parquet con Spark
try:
    if not os.path.exists(PARQUET_SOURCE_PATH):
        raise FileNotFoundError(f"No se encontró el archivo en {PARQUET_SOURCE_PATH}")
        
    df_manifold = spark.read.parquet(PARQUET_SOURCE_PATH)

    # 3. Registrar como vista temporal para SQL
    df_manifold.createOrReplaceTempView("simulacion_manifold")

    print("--- ¡ÉXITO! MANIFOLD V8 CARGADO ---")
    print(f"📊 Total de filas: {df_manifold.count():,}")
    df_manifold.printSchema()

except Exception as e:
    print(f"❌ ERROR: No se pudo leer el archivo. Verifique la ruta.")
    print(e)

📂 Cargando dataset maestro desde: /workspaces/pienza/data/dumped_files/260426_cGAN_manifold_v8.parquet ...


--- ¡ÉXITO! MANIFOLD V8 CARGADO ---
📊 Total de filas: 1,010,001
root
 |-- upfront_fare: float (nullable = true)
 |-- est_trip_time_sec: float (nullable = true)
 |-- est_trip_dist_km: float (nullable = true)
 |-- time_to_pickup_sec: float (nullable = true)
 |-- dist_to_pickup_km: float (nullable = true)
 |-- hour_of_day: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- product_category_fk: string (nullable = true)
 |-- dropoff_zone_id: string (nullable = true)
 |-- pickup_zone_id: string (nullable = true)
 |-- reason_primary_fk: string (nullable = true)



In [5]:
# ==============================================================================
# CELL 5: CONFIGURACIÓN DE DESTINO BIGQUERY (PIENZA_BIG)
# ==============================================================================
# Purpose: Prepare Spark for BigQuery writing using the GCS staging bucket.
# ==============================================================================

# 1. Variables de proyecto
GCP_PROJECT_ID = "drivers-dilemma" 
BQ_DATASET     = "pienza_big"      

# 2. Configuración de Staging en GCS
# Spark necesita un 'cubeta' temporal para mover los datos antes de inyectarlos a BQ
STAGING_BUCKET = "pienza-streamlit" # Usamos tu bucket ya validado

spark.conf.set("temporaryGcsBucket", STAGING_BUCKET)

print(f"✅ Arquitectura Cloud configurada:")
print(f"   📍 Destino: {GCP_PROJECT_ID}.{BQ_DATASET}")
print(f"   🪣  Staging GCS: {STAGING_BUCKET}")

✅ Arquitectura Cloud configurada:
   📍 Destino: drivers-dilemma.pienza_big
   🪣  Staging GCS: pienza-streamlit


In [6]:
# ==============================================================================
# CELL 6: THE UNIVERSAL LOOKUP TABLE (BULLETPROOF VERSION)
# ==============================================================================
from pyspark.sql import functions as F

print("🏗️  Forjando la Piedra Rosetta de Pienza (Protocolo de Máxima Robustez)...")

# --- 1. CONFIGURACIÓN DE IDENTIDAD (Asegurar ID de Texto) ---
# Usamos el ID de texto, que es el estándar para recursos en GCP
BILLING_PROJECT_ID = "drivers-dilemma"
DATASET_REALIDAD = "pienza_mini"

# --- 2. DICCIONARIO HUMANO (P_ Axis) ---
salchichota_data = [
    (0, "santa_fe_bosques_de__santa_fe_cumbres_de__santa_fe_tec"),
    (1, "santa_fe_centro_comercial"),
    (2, "carretera_al_olivo__carretera_libre__cruce_echanove__vistahermosa"),
    (3, "bosques_pabellon__el_olivo__loma_de_la_palma"),
    (4, "agwa_bezares__reforma_bnp"), (5, "ahuehuetes_norte__de_los_bosques"),
    (6, "interlomas_haciendas__jesus_del_monte"), (7, "blvrd_anahuac__universidad_anahuac"),
    (8, "ave_club_de_golf_lomas__interlomas_magnocentro__vialidad_de_la_barranca"),
    (9, "santa_fe_ibero"), (10, "lomas_altas__nodo_reforma_palmas__reforma_regina"),
    (11, "ahuehuetes_sur"), (12, "tamarindos"), (13, "santa_fe_quintana__sante_fe_patio"),
    (14, "bosque_real__lomas_country_club"), (15, "herradura_conscripto"),
    (16, "de_las_fuentes__tecamachalco"), (17, "lomas_barrilaco__lomas_olimpo__nodo_monte_libano"),
    (18, "lomas_prado_norte__lomas_trastevere"), (19, "lomas_virreyes"),
    (20, "lomas_fc_cuernavaca"), (21, "fuentes_casino__sedena__tecamachalco"),
    (22, "palmas_jp_morgan"), (23, "bondojito_asf__bosque_2__bosque_3"),
    (24, "bosque_1__campo_marte"), (25, "roma_condesa_2"), (26, "rios"),
    (27, "roma_condesa_1"), (28, "juarez_soho_house"), (29, "anzures"),
    (30, "anahuac_1__bahias__frontera_polanco"), (31, "sotelo"),
    (32, "carso_antara_miyana"), (33, "irrigacion__polanco_uber_hq"),
    (34, "juarez_rosa"), (35, "lagos"), (36, "polanco_grupo_mexico__polanco_palacio"),
    (37, "polanco_gandhi"), (38, "polanco_5"), (39, "polanco_parque_lincoln"),
    (40, "polanco_parroquia"), (41, "santa_fe_colegios")
]
df_dict_p = spark.createDataFrame(salchichota_data, ["id_raw", "semantic_name"]) \
                 .withColumn("zone_key", F.concat(F.lit("P_"), F.col("id_raw").cast("string"))) \
                 .select("zone_key", "semantic_name")

# --- 3. DICCIONARIO MÁQUINA (C_ Axis) ---
try:
    print(f"📡 Recuperando nombres de clusters desde {DATASET_REALIDAD}.silver_palette...")
    # Agregamos parentProject para que Google sepa a quién cobrar los bytes escaneados
    df_silver_palette = spark.read.format("bigquery") \
        .option("table", f"{BILLING_PROJECT_ID}.{DATASET_REALIDAD}.silver_palette") \
        .option("parentProject", BILLING_PROJECT_ID) \
        .load()

    df_dict_c = df_silver_palette \
        .select("dropoff_hdbscan_id", "dropoff_hdbscan_name") \
        .distinct() \
        .filter(F.col("dropoff_hdbscan_id") != -1) \
        .withColumn("zone_key", F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string"))) \
        .select("zone_key", F.col("dropoff_hdbscan_name").alias("semantic_name"))

    print(f"   ✅ Metadatos de máquina recuperados: {df_dict_c.count()} clústeres.")

except Exception as e:
    print("🔴 FALLO CRÍTICO AL LEER BIGQUERY.")
    print("Posibles causas: El nombre 'drivers-dilemma' es incorrecto o falta el parentProject.")
    raise e

# --- 4. UNIFICACIÓN FINAL ---
df_unassigned = spark.createDataFrame([("Unassigned", "unassigned_area")], ["zone_key", "semantic_name"])

# Unión Triple
df_master_dictionary = df_dict_p.union(df_dict_c).union(df_unassigned)

print(f"✅ Diccionario maestro unificado: {df_master_dictionary.count()} zonas.")
df_master_dictionary.orderBy("zone_key").show(10, truncate=False)

🏗️  Forjando la Piedra Rosetta de Pienza (Protocolo de Máxima Robustez)...
📡 Recuperando nombres de clusters desde pienza_mini.silver_palette...


   ✅ Metadatos de máquina recuperados: 46 clústeres.


✅ Diccionario maestro unificado: 89 zonas.
+--------+-----------------------+
|zone_key|semantic_name          |
+--------+-----------------------+
|C_-2    |missing_coordinates    |
|C_0     |viaducto_tlalpan       |
|C_1     |terminal_1_aicm        |
|C_10    |observatorio           |
|C_11    |sotelo_san_esteban     |
|C_12    |barranca_del_muerto    |
|C_13    |haciendas_san_fernando |
|C_14    |vialidad_de_la_barranca|
|C_15    |interlomas_magnocentro |
|C_16    |santa_fe_itesm         |
+--------+-----------------------+
only showing top 10 rows



In [7]:
# ==============================================================================
# CELL 6.1: FULL GEOGRAPHIC RECONNAISSANCE (THE 89 ZONES)
# ==============================================================================
# Purpose: Display the complete Pienza dictionary for manual audit.
# ==============================================================================

print("📖 LISTADO MAESTRO DE ZONAS (89 IDENTIDADES SEMÁNTICAS)")
print("-" * 80)

# Convertimos a Pandas para visualización premium en Colab
df_full_audit = df_master_dictionary.orderBy("zone_key").toPandas()

# Configuramos Pandas para que no trunque los nombres largos de la "Salchichota"
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)

display(df_full_audit)

print("-" * 80)
print(f"✅ Total: {len(df_full_audit)} registros listos para el JOIN del millón.")

📖 LISTADO MAESTRO DE ZONAS (89 IDENTIDADES SEMÁNTICAS)
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_0,viaducto_tlalpan
2,C_1,terminal_1_aicm
3,C_10,observatorio
4,C_11,sotelo_san_esteban
5,C_12,barranca_del_muerto
6,C_13,haciendas_san_fernando
7,C_14,vialidad_de_la_barranca
8,C_15,interlomas_magnocentro
9,C_16,santa_fe_itesm


--------------------------------------------------------------------------------
✅ Total: 89 registros listos para el JOIN del millón.


In [8]:
# ==============================================================================
# CELL 6.2: THE GEOGRAPHIC PURGE (NON-DESTRUCTIVE)
# ==============================================================================

print("🧹 Identificando zonas activas en el Manifold...")

# 1. Guardamos una copia del diccionario completo (89 filas) antes de filtrar
df_master_dictionary_full = df_master_dictionary

# 2. Identificar los IDs que REALMENTE existen en el Manifold (1M de filas)
df_active_keys = df_manifold.select("dropoff_zone_id").distinct()

# 3. Crear el diccionario curado (67 filas)
df_dict_curated = df_master_dictionary_full.join(
    df_active_keys,
    df_master_dictionary_full.zone_key == df_active_keys.dropoff_zone_id,
    "inner"
).select("zone_key", "semantic_name")

print(f"   ✅ Filtrado completado: 89 -> {df_dict_curated.count()} zonas vivas.")

# Sobrescribimos el puntero principal para el JOIN final
df_master_dictionary = df_dict_curated

🧹 Identificando zonas activas en el Manifold...


   ✅ Filtrado completado: 89 -> 67 zonas vivas.


In [9]:
# ==============================================================================
# CELL 6.3: THE SHADOW AUDIT (IDENTIFYING DELETED ZONES)
# ==============================================================================

print("🕵️‍♂️ Investigando las 22 zonas eliminadas...")

# Buscamos lo que está en el FULL pero NO en el CURATED
df_deleted_zones = df_master_dictionary_full.join(
    df_active_keys,
    df_master_dictionary_full.zone_key == df_active_keys.dropoff_zone_id,
    "left_anti"
)

# Convertimos a Pandas para el reporte
df_deleted_report = df_deleted_zones.orderBy("zone_key").toPandas()

print(f"\n🚫 LISTADO DE EXCLUSIÓN (Zonas Sombra / Absorbidas):")
print("-" * 80)
if len(df_deleted_report) > 0:
    display(df_deleted_report)
else:
    print("✨ No se encontraron zonas eliminadas.")

print("-" * 80)
print(f"✅ Auditoría terminada. Estas zonas ya no contaminarán el Manifold.")

🕵️‍♂️ Investigando las 22 zonas eliminadas...

🚫 LISTADO DE EXCLUSIÓN (Zonas Sombra / Absorbidas):
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_14,vialidad_de_la_barranca
2,C_20,tamarindos
3,C_21,nodo_constituyentes_reforma
4,C_24,santa_fe_core
5,C_25,pabellon_bosques
6,C_26,duraznos
7,C_27,chamizal
8,C_28,lomas_anahuac
9,C_31,san_miguel_chapultepec


--------------------------------------------------------------------------------
✅ Auditoría terminada. Estas zonas ya no contaminarán el Manifold.


In [10]:
# ==============================================================================
# CELL 6.4: THE GHOST RECOVERY (FORCED COMPARISON)
# ==============================================================================

print("🕵️‍♂️  Recuperando los 22 fantasmas desde los cimientos...")

# 1. Reconstruimos el diccionario completo (P_ + C_)
df_reconstructed_full = df_dict_p.union(df_dict_c).union(df_unassigned)

# 2. Identificamos los IDs que realmente están en el Manifold (1M)
df_active_manifold_keys = df_manifold.select("dropoff_zone_id").distinct()

# 3. ANTI-JOIN: ¿Qué estaba en el diccionario pero NO llegó al Manifold?
df_ghosts = df_reconstructed_full.join(
    df_active_manifold_keys,
    df_reconstructed_full.zone_key == df_active_manifold_keys.dropoff_zone_id,
    "left_anti"
)

# 4. Reporte Final
ghosts_pd = df_ghosts.orderBy("zone_key").toPandas()

print(f"\n🚫  ZONAS PURGADAS ({len(ghosts_pd)} registros):")
print("-" * 80)
if len(ghosts_pd) > 0:
    display(ghosts_pd)
    print("\n💡 ANÁLISIS SOBERANO: Estas zonas fueron absorbidas por tus polígonos P_")
    print("   o el GAN determinó que su probabilidad de ocurrencia es cero.")
else:
    print("🔎 Algo sigue mal en la referencia. Procedamos al JOIN para no perder el momentum.")

# Aseguramos que para el JOIN final usamos la versión curada
df_master_dictionary = df_reconstructed_full.join(
    df_active_manifold_keys,
    df_reconstructed_full.zone_key == df_active_manifold_keys.dropoff_zone_id,
    "inner"
).select("zone_key", "semantic_name")

🕵️‍♂️  Recuperando los 22 fantasmas desde los cimientos...

🚫  ZONAS PURGADAS (22 registros):
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_14,vialidad_de_la_barranca
2,C_20,tamarindos
3,C_21,nodo_constituyentes_reforma
4,C_24,santa_fe_core
5,C_25,pabellon_bosques
6,C_26,duraznos
7,C_27,chamizal
8,C_28,lomas_anahuac
9,C_31,san_miguel_chapultepec



💡 ANÁLISIS SOBERANO: Estas zonas fueron absorbidas por tus polígonos P_
   o el GAN determinó que su probabilidad de ocurrencia es cero.


In [11]:
# ==============================================================================
# CELL 6.5: REAL-WORLD COMPOSITION AUDIT (THE 67 SURVIVORS - BUG FIX)
# ==============================================================================
from pyspark.sql import functions as F
from itertools import chain

print("📡 Iniciando Auditoría Forense de la Realidad...")

# --- 1. RE-INYECCIÓN DEL MAPA CANÓNICO ---
id_map = {
    -1: -1, 41: 0, 42: 0, 46: 0, 43: 1, 65: 2, 62: 2, 44: 2, 36: 2, 49: 3, 52: 3,
    35: 3, 50: 4, 58: 4, 25: 5, 31: 5, 63: 6, 39: 6, 51: 7, 33: 7, 37: 8, 53: 8,
    48: 8, 60: 9, 57: 10, 12: 10, 32: 10, 24: 11, 40: 12, 45: 13, 59: 13, 61: 14,
    38: 14, 34: 15, 30: 16, 66: 16, 17: 17, 14: 17, 22: 17, 16: 18, 13: 18, 11: 19,
    15: 20, 21: 21, 20: 21, 19: 21, 18: 22, 47: 23, 55: 23, 56: 23, 54: 24, 64: 24,
    71: 25, 9: 26, 70: 27, 69: 28, 8: 29, 6: 30, 7: 30, 23: 30, 3: 31, 2: 32,
    4: 33, 29: 33, 68: 34, 5: 35, 27: 36, 28: 36, 1: 37, 10: 38, 0: 39, 26: 40, 67: 41
}

try:
    # --- 2. CARGA DESDE BIGQUERY ---
    df_real_raw = spark.read.format("bigquery") \
        .option("parentProject", BILLING_PROJECT_ID) \
        .option("table", f"{BILLING_PROJECT_ID}.{DATASET_REALIDAD}.v_ML_Supervised") \
        .option("viewsEnabled", "true") \
        .option("materializationDataset", DATASET_REALIDAD) \
        .load() \
        .select("offer_id", "dropoff_polygon_id", "dropoff_hdbscan_id")

    # --- 3. PROCESAMIENTO SPATIAL ---
    # Convertimos id_map a una columna tipo Map de Spark
    mapping_expr = F.create_map([F.lit(x) for x in chain(*id_map.items())])

    # Aplicamos Coalesce y Mapeo (Sintaxis Spark 3.0+)
    df_real_mapped = df_real_raw.withColumn(
        "id_agrupado",
        F.coalesce(mapping_expr[F.col("dropoff_polygon_id").cast("int")], F.lit(-1))
    )

    # Construimos la llave zone_key
    df_real_final = df_real_mapped.withColumn("zone_key",
        F.when(F.col("id_agrupado") >= 0, F.concat(F.lit("P_"), F.col("id_agrupado").cast("string")))
         .when(F.col("dropoff_hdbscan_id") > -1, F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string")))
         .otherwise("Unassigned")
    )

    # Conteo de frecuencia real
    df_real_counts = df_real_final.groupBy("zone_key").count().withColumnRenamed("count", "n_obs_real")

    # --- 4. JOIN Y LIMPIEZA DE NULOS (FIXED) ---
    # Usamos F.coalesce para llenar los nulos en la columna resultante del join
    df_final_audit_table = df_master_dictionary.join(
        df_real_counts,
        "zone_key",
        "left"
    ).select(
        "zone_key",
        "semantic_name",
        F.coalesce(F.col("n_obs_real"), F.lit(0)).alias("n_obs_real")
    ).orderBy(F.col("n_obs_real").desc())

    # --- 5. REPORTE FINAL ---
    print("\n🏆 CENSO REALIDAD (67 ZONAS VIVAS):")
    final_report_pd = df_final_audit_table.toPandas()
    display(final_report_pd)

    print("-" * 80)
    print(f"✅ Auditoría completada. El ADN real está sincronizado.")

except Exception as e:
    print(f"🔴 ERROR EN LA AUDITORÍA: {e}")
    raise e

📡 Iniciando Auditoría Forense de la Realidad...

🏆 CENSO REALIDAD (67 ZONAS VIVAS):


,zone_key,semantic_name,n_obs_real
0,Unassigned,unassigned_area,1366
1,P_25,roma_condesa_2,223
2,P_1,santa_fe_centro_comercial,169
3,P_9,santa_fe_ibero,150
4,P_32,carso_antara_miyana,122
5,C_1,terminal_1_aicm,111
6,P_13,santa_fe_quintana__sante_fe_patio,104
7,P_15,herradura_conscripto,99
8,P_8,ave_club_de_golf_lomas__interlomas_magnocentro__vialidad_de_la_barranca,96
9,P_12,tamarindos,82


--------------------------------------------------------------------------------
✅ Auditoría completada. El ADN real está sincronizado.


### 📝 Nota Técnica: Auditoría de Residuales Topológicos (P_ vs. C_)

Tras ejecutar el censo de densidad en la realidad (**Cell 6.5**), se identificaron 6 zonas de origen algorítmico (**C_**) con un conteo de observaciones inusualmente bajo ($N < 25$), destacando casos como `C_18 (santa_fe_patio)` y `C_32 (tacubaya)` con $N=1$.

**Análisis de la Anomalía:**
Aunque el objetivo inicial del *Master Coalesce* era que los polígonos humanos absorbierean la mayor parte de la actividad, estos residuales representan un fenómeno de **"Spillover Geo-Estadístico"**:

1.  **Límites Duros vs. Densidad Suave:** Los polígonos manuales tienen fronteras geométricas rígidas. HDBSCAN, por el contrario, agrupa por densidad. Estos puntos son ofertas que el algoritmo detectó como parte del "clúster" (ej. el ecosistema del ITESM), pero que físicamente cayeron a unos metros de distancia, fuera del trazo manual del polígono.
2.  **Artefactos Naturales:** Casos como `santa_fe_itesm` ($N=6$) o `santa_fe_abc` ($N=8$) son críticos. Representan la "periferia" de zonas de alta demanda que la máquina reconoce aunque el ojo humano no las haya encerrado.

**Decisión Arquitectónica:**
Se ha decidido **PRESERVAR** estas 67 zonas (incluyendo los residuales) en el Manifold por dos razones:
*   **Fidelidad del Generador:** El GAN v2 aprendió la existencia de estos micro-puntos. Eliminarlos ahora crearía una inconsistencia entre el modelo entrenado y el diccionario de salida.
*   **Rigor Forense:** Estos puntos representan la "frontera" real del mercado. En la simulación de la Fase VI, estas zonas actuarán como *edge cases* naturales, permitiendo testear la sensibilidad del modelo en zonas de baja visibilidad.

**Visto Bueno (VoBo):** La topología se considera finalizada y validada. Procediendo a la denormalización masiva del millón de registros.

In [13]:
# ==============================================================================
# CELL 6.7: THE UNIFIED DICTIONARY FORGE & PERSISTENCE (v2.0)
# ==============================================================================
# Purpose: Manually forge and persist both Product and Geographic dictionaries.
# Environment: Codespaces Local Workspace (Sovereign Mode).
# Assets: dim_product_hierarchy.parquet & dim_dropoff_zone_dictionary.parquet
# ==============================================================================
from pyspark.sql import functions as F
import os

# --- 1. CONFIGURACIÓN DE RUTAS LOCALES ---
DICT_DIR = "/workspaces/pienza/data/dictionaries"
os.makedirs(DICT_DIR, exist_ok=True) # Asegurar que la carpeta existe

PROD_DICT_PATH = f"{DICT_DIR}/dim_product_hierarchy.parquet"
ZONE_DICT_PATH = f"{DICT_DIR}/dim_dropoff_zone_dictionary.parquet"

print("🏗️  FORJANDO BÓVEDA DE METADATOS PIENZA (LOCAL)...")
print("-" * 65)

# --- 2. DICCIONARIO DE PRODUCTOS (JERARQUÍA 1-2-3) ---
products_dna = [(1, "X"), (2, "Mid-Tier"), (3, "Premium")]
df_dim_products = spark.createDataFrame(products_dna, ["id_p", "product_name"])

# --- 3. DICCIONARIO DE GEOGRAFÍA (SALCHICHOTA + SURVIVORS) ---
# A. Nombres Manuales (P_ Axis)
salchichota_data = [
    (0, "santa_fe_bosques_de__santa_fe_cumbres_de__santa_fe_tec"),
    (1, "santa_fe_centro_comercial"),
    (2, "carretera_al_olivo__carretera_libre__cruce_echanove__vistahermosa"),
    (3, "bosques_pabellon__el_olivo__loma_de_la_palma"),
    (4, "agwa_bezares__reforma_bnp"), (5, "ahuehuetes_norte__de_los_bosques"),
    (6, "interlomas_haciendas__jesus_del_monte"), (7, "blvrd_anahuac__universidad_anahuac"),
    (8, "ave_club_de_golf_lomas__interlomas_magnocentro__vialidad_de_la_barranca"),
    (9, "santa_fe_ibero"), (10, "lomas_altas__nodo_reforma_palmas__reforma_regina"),
    (11, "ahuehuetes_sur"), (12, "tamarindos"), (13, "santa_fe_quintana__sante_fe_patio"),
    (14, "bosque_real__lomas_country_club"), (15, "herradura_conscripto"),
    (16, "de_las_fuentes__tecamachalco"), (17, "lomas_barrilaco__lomas_olimpo__nodo_monte_libano"),
    (18, "lomas_prado_norte__lomas_trastevere"), (19, "lomas_virreyes"),
    (20, "lomas_fc_cuernavaca"), (21, "fuentes_casino__sedena__tecamachalco"),
    (22, "palmas_jp_morgan"), (23, "bondojito_asf__bosque_2__bosque_3"),
    (24, "bosque_1__campo_marte"), (25, "roma_condesa_2"), (26, "rios"),
    (27, "roma_condesa_1"), (28, "juarez_soho_house"), (29, "anzures"),
    (30, "anahuac_1__bahias__frontera_polanco"), (31, "sotelo"),
    (32, "carso_antara_miyana"), (33, "irrigacion__polanco_uber_hq"),
    (34, "juarez_rosa"), (35, "lagos"), (36, "polanco_grupo_mexico__polanco_palacio"),
    (37, "polanco_gandhi"), (38, "polanco_5"), (39, "polanco_parque_lincoln"),
    (40, "polanco_parroquia"), (41, "santa_fe_colegios")
]
df_p = spark.createDataFrame(salchichota_data, ["id", "semantic_name"]) \
            .withColumn("zone_key", F.concat(F.lit("P_"), F.col("id").cast("string"))) \
            .select("zone_key", "semantic_name")

# B. Nombres Máquina (C_ Axis desde BigQuery Realidad)
print("📡 Recuperando metadatos de clústeres desde BigQuery...")
try:
    df_c = spark.read.format("bigquery") \
            .option("table", f"{GCP_PROJECT_ID}.pienza_mini.silver_palette") \
            .option("parentProject", GCP_PROJECT_ID) \
            .load() \
            .select("dropoff_hdbscan_id", "dropoff_hdbscan_name") \
            .distinct() \
            .filter(F.col("dropoff_hdbscan_id") != -1) \
            .withColumn("zone_key", F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string"))) \
            .select("zone_key", F.col("dropoff_hdbscan_name").alias("semantic_name"))
except Exception as e:
    print(f"⚠️ Error al conectar con BQ: {e}. Usando fallback vacío para C_ Axis.")
    df_c = spark.createDataFrame([], df_p.schema)

# C. Unassigned & Unión
df_u = spark.createDataFrame([("Unassigned", "unassigned_area")], ["zone_key", "semantic_name"])
df_full_dict = df_p.union(df_c).union(df_u)

# D. Filtro de Supervivencia (Solo las 67 que existen en el Manifold V8)
print("🧹 Filtrando por supervivencia en el Manifold...")
df_active_keys = df_manifold.select("dropoff_zone_id").distinct()
df_dim_zones = df_full_dict.join(df_active_keys, df_full_dict.zone_key == df_active_keys.dropoff_zone_id, "inner") \
                           .select("zone_key", "semantic_name")

# --- 4. PERSISTENCIA FÍSICA EN WORKSPACE ---
try:
    print(f"💾 Guardando diccionarios en: {DICT_DIR} ...")
    df_dim_products.toPandas().to_parquet(PROD_DICT_PATH, index=False)
    df_dim_zones.toPandas().to_parquet(ZONE_DICT_PATH, index=False)

    print("\n✅ BÓVEDA DE METADATOS ACTUALIZADA.")
    print(f"   - Productos: {df_dim_products.count()} niveles (DNA de Pienza)")
    print(f"   - Geografía: {df_dim_zones.count()} zonas (Salchichota + Clústeres)")
except Exception as e:
    print(f"❌ Error en la persistencia local: {e}")

print("-" * 65)

🏗️  FORJANDO BÓVEDA DE METADATOS PIENZA (LOCAL)...
-----------------------------------------------------------------
📡 Recuperando metadatos de clústeres desde BigQuery...
🧹 Filtrando por supervivencia en el Manifold...
💾 Guardando diccionarios en: /workspaces/pienza/data/dictionaries ...

✅ BÓVEDA DE METADATOS ACTUALIZADA.
   - Productos: 3 niveles (DNA de Pienza)
   - Geografía: 67 zonas (Salchichota + Clústeres)
-----------------------------------------------------------------


In [14]:
# ==============================================================================
# CELL 6.8: THE PICKUP DICTIONARY FORGE (MIRROR OF DROPOFF)
# ==============================================================================
# Purpose: Create the specific dictionary for Origin Zones (Pickup).
# Logic:   Master Dict (P_ + C_) filtered by Manifold's 'pickup_zone_id'.
# Assets:  dim_pickup_zone_dictionary.parquet
# ==============================================================================

# --- 1. CONFIGURACIÓN DE RUTA LOCAL ---
# Usamos el mismo directorio definido en la Cell 6.7
PICKUP_DICT_PATH = "/workspaces/pienza/data/dictionaries/dim_pickup_zone_dictionary.parquet"

print("🏗️  FORJANDO DICCIONARIO DE PICKUP (ORIGEN)...")
print("-" * 65)

# --- 2. RECONSTRUCCIÓN DEL UNIVERSO (Protocolo de Seguridad) ---
# Si reiniciaste el kernel, estas variables podrían no estar en memoria.
if 'df_full_dict' not in locals():
    print("   -> Reconstruyendo Universo de Zonas (P_ + C_) desde variables de sesión...")
    # Asumimos que df_p, df_c y df_u fueron creados en la celda 6.7
    try:
        df_full_dict = df_p.union(df_c).union(df_u)
    except NameError:
        print("❌ ERROR CRÍTICO: Debe ejecutar la CELL 6.7 primero para definir los ejes P_ y C_.")
        raise

# --- 3. FILTRO DE SUPERVIVENCIA (PICKUP) ---
print("🧹 Identificando zonas de origen activas en el Manifold...")

# Verificamos que la columna exista en el dataset cargado en la Cell 4
if "pickup_zone_id" in df_manifold.columns:
    # Extraemos IDs únicos de origen del millón de filas
    df_active_pickup = df_manifold.select("pickup_zone_id").distinct()

    # JOIN: Mapeamos el universo total contra los IDs reales de origen
    df_dim_pickup = df_full_dict.join(
        df_active_pickup,
        df_full_dict.zone_key == df_active_pickup.pickup_zone_id,
        "inner"
    ).select("zone_key", "semantic_name")

    # --- 4. PERSISTENCIA EN EL WORKSPACE ---
    try:
        print(f"💾 Guardando diccionario de origen en: {PICKUP_DICT_PATH}")
        # Convertimos a Pandas para escritura rápida en formato Parquet local
        df_dim_pickup.toPandas().to_parquet(PICKUP_DICT_PATH, index=False)

        print(f"\n✅ DICCIONARIO PICKUP ASEGURADO.")
        print(f"   📍 Total: {df_dim_pickup.count()} identidades semánticas de origen.")
        print(f"   🚀 Listo para el Triple Star Schema Join.")
    except Exception as e:
        print(f"❌ Error en la persistencia local: {e}")

else:
    print("❌ ERROR CRÍTICO: No se encontró 'pickup_zone_id' en el Manifold.")
    print("   Columnas detectadas:", df_manifold.columns)

print("-" * 65)

🏗️  FORJANDO DICCIONARIO DE PICKUP (ORIGEN)...
-----------------------------------------------------------------
🧹 Identificando zonas de origen activas en el Manifold...
💾 Guardando diccionario de origen en: /workspaces/pienza/data/dictionaries/dim_pickup_zone_dictionary.parquet

✅ DICCIONARIO PICKUP ASEGURADO.
   📍 Total: 43 identidades semánticas de origen.
   🚀 Listo para el Triple Star Schema Join.
-----------------------------------------------------------------


In [15]:
# ==============================================================================
# CELL 7: THE TRIPLE STAR SCHEMA JOIN (CONSUMER MODE)
# ==============================================================================
# Purpose: Denormalize the Manifold V8 using validated local dictionaries.
# Logic:   Triple Broadcast Join (High Performance) to inject semantic names.
# ==============================================================================
from pyspark.sql import functions as F

print("🚀 Inyectando Inteligencia Semántica al Manifold Sintético...")
print("-" * 80)

# --- 1. RUTAS DE ARTEFACTOS LOCALES (Codespaces) ---
DICT_DIR      = "/workspaces/pienza/data/dictionaries"
PATH_MANIFOLD = "/workspaces/pienza/data/dumped_files/260426_cGAN_manifold_v8.parquet"

PATH_DIM_PROD = f"{DICT_DIR}/dim_product_hierarchy.parquet"
PATH_DIM_DROP = f"{DICT_DIR}/dim_dropoff_zone_dictionary.parquet"
PATH_DIM_PICK = f"{DICT_DIR}/dim_pickup_zone_dictionary.parquet"

# --- 2. LECTURA DE DATOS ---
print("   -> Cargando Diccionarios y Manifold...")
try:
    df_dim_prods = spark.read.parquet(PATH_DIM_PROD)
    df_dim_drop  = spark.read.parquet(PATH_DIM_DROP)
    df_dim_pick  = spark.read.parquet(PATH_DIM_PICK)
    
    df_facts = spark.read.parquet(PATH_MANIFOLD)
    print(f"   ✅ Datos cargados. Manifold detectado con {df_facts.count():,} registros.")

except Exception as e:
    print("❌ ERROR: No se encontraron los archivos locales. Verifique las celdas 4, 6.7 y 6.8.")
    raise e

# --- 3. TRIPLE BROADCAST JOIN (Optimización de Memoria) ---
# Usamos broadcast porque los diccionarios son pequeños y el manifold es masivo.
print("⚡ Ejecutando Desnormalización Masiva (3 Dimensiones)...")

# A. Dimensión: Producto (X, Mid, Premium)
df_step1 = df_facts.join(
    F.broadcast(df_dim_prods),
    df_facts.product_category_fk == df_dim_prods.id_p,
    "left"
).drop("id_p")

# B. Dimensión: Dropoff (Destino)
df_step2 = df_step1.join(
    F.broadcast(df_dim_drop),
    df_step1.dropoff_zone_id == df_dim_drop.zone_key,
    "left"
).drop("zone_key").withColumnRenamed("semantic_name", "dropoff_name")

# C. Dimensión: Pickup (Origen)
df_final = df_step2.join(
    F.broadcast(df_dim_pick),
    df_step2.pickup_zone_id == df_dim_pick.zone_key,
    "left"
).drop("zone_key").withColumnRenamed("semantic_name", "pickup_name")

# --- 4. VALIDACIÓN DE COBERTURA ---
print("\n🧪 AUDITORÍA DE INTEGRIDAD SEMÁNTICA:")
print("-" * 80)

total = df_final.count()
named_pickup  = df_final.filter(F.col("pickup_name").isNotNull()).count()
named_dropoff = df_final.filter(F.col("dropoff_name").isNotNull()).count()

print(f"📊 Cobertura Pickup:  {named_pickup:,} / {total:,} ({(named_pickup/total)*100:.2f}%)")
print(f"📊 Cobertura Dropoff: {named_dropoff:,} / {total:,} ({(named_dropoff/total)*100:.2f}%)")

# Muestra aleatoria para deleite visual
print("\n✨ MUESTRA DEL MANIFOLD ENRIQUECIDO:")
df_final.select(
    "product_name", 
    "pickup_name", 
    "dropoff_name", 
    "upfront_fare"
).filter(F.col("pickup_name").isNotNull()) \
 .orderBy(F.rand()) \
 .show(15, truncate=False)

# --- 5. PERSISTENCIA EN CACHÉ ---
# Guardamos en memoria para las auditorías "bien padre" y la subida final
df_final.cache()
print("🏁 ESTATUS: EL DATASET MAESTRO ESTÁ LISTO EN EL MOTOR DE SPARK.")

🚀 Inyectando Inteligencia Semántica al Manifold Sintético...
--------------------------------------------------------------------------------
   -> Cargando Diccionarios y Manifold...
   ✅ Datos cargados. Manifold detectado con 1,010,001 registros.
⚡ Ejecutando Desnormalización Masiva (3 Dimensiones)...

🧪 AUDITORÍA DE INTEGRIDAD SEMÁNTICA:
--------------------------------------------------------------------------------


📊 Cobertura Pickup:  1,010,001 / 1,010,001 (100.00%)
📊 Cobertura Dropoff: 1,010,001 / 1,010,001 (100.00%)

✨ MUESTRA DEL MANIFOLD ENRIQUECIDO:


+------------+-----------------------------------------------------------------------+-----------------------------------------------------------------------+------------+
|product_name|pickup_name                                                            |dropoff_name                                                           |upfront_fare|
+------------+-----------------------------------------------------------------------+-----------------------------------------------------------------------+------------+
|X           |bosques_pabellon__el_olivo__loma_de_la_palma                           |lomas_fc_cuernavaca                                                    |120.04237   |
|X           |santa_fe_centro_comercial                                              |unassigned_area                                                        |258.5337    |
|X           |polanco_parque_lincoln                                                 |ave_club_de_golf_lomas__interlomas_magnocentro__vialid

In [16]:
# ==============================================================================
# CELL 9 (FINAL - NATIVE): BYPASSING JAVA/SPARK HADOOP ERRORS
# ==============================================================================
# Purpose: Convert to Pandas and upload via BigQuery Python Client (Native API).
# Logic:   Leveraging local RAM for a high-speed data injection to the Cloud.
# ==============================================================================
from google.cloud import bigquery
import pandas as pd

# 1. Bajar de Spark a Pandas 
# Nota: Aprovechamos la RAM de Codespaces para manejar el millón de registros.
print("📦 Convirtiendo Manifold a Pandas (Bajando de Spark)...")
try:
    pdf_final = df_final.toPandas()
    print(f"   ✅ Conversión exitosa. DataFrame de Pandas listo con {len(pdf_final):,} filas.")
except Exception as e:
    print("❌ ERROR en la conversión a Pandas. Verifique la memoria disponible.")
    raise e

# 2. Configuración de Destino (Usando variables de Cell 5)
TARGET_TABLE = "synthetic_manifold_v8_enriched"
FULL_TABLE_ID = f"{GCP_PROJECT_ID}.{BQ_DATASET}.{TARGET_TABLE}"

# 3. Inicializar Cliente Nativo
# El cliente detectará automáticamente la Service Account configurada en Cell 3.
client = bigquery.Client(project=GCP_PROJECT_ID)

# 4. Configuración del Job de Carga
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE", # Sobrescribir para mantener el 'Canon' fresco
)

print(f"📡 Iniciando inyección masiva a BigQuery: {FULL_TABLE_ID} ...")

try:
    # Subida directa mediante el cliente de Python
    # Este método es más resiliente que el conector de Spark para entornos de Codespaces.
    job = client.load_table_from_dataframe(
        pdf_final, FULL_TABLE_ID, job_config=job_config
    )
    
    print("⏳ Procesando carga en la nube (esto puede tomar un minuto)...")
    job.result()  # Esperar a que BigQuery certifique la carga

    print(f"\n🚀 ¡MISIÓN CUMPLIDA! El Manifold V8 Enriquecido ya vive en BigQuery.")
    print(f"📍 Tabla Soberana: {FULL_TABLE_ID}")
    print("-" * 80)
    print("🏆 ESTATUS: INFRAESTRUCTURA GENERATIVA COMPLETADA AL 100%.")

except Exception as e:
    print("❌ ERROR CRÍTICO EN SUBIDA NATIVA:")
    print(e)

📦 Convirtiendo Manifold a Pandas (Bajando de Spark)...


   ✅ Conversión exitosa. DataFrame de Pandas listo con 1,010,001 filas.
📡 Iniciando inyección masiva a BigQuery: drivers-dilemma.pienza_big.synthetic_manifold_v8_enriched ...
⏳ Procesando carga en la nube (esto puede tomar un minuto)...

🚀 ¡MISIÓN CUMPLIDA! El Manifold V8 Enriquecido ya vive en BigQuery.
📍 Tabla Soberana: drivers-dilemma.pienza_big.synthetic_manifold_v8_enriched
--------------------------------------------------------------------------------
🏆 ESTATUS: INFRAESTRUCTURA GENERATIVA COMPLETADA AL 100%.


In [17]:
# ==============================================================================
# AUDITORÍA "BIEN PADRE": EL CORAZÓN DEL MANIFOLD (VIAJES ENDÓGENOS)
# ==============================================================================
# Propósito: Ver cuántos viajes ocurren 100% dentro de tus zonas maestras.
# ==============================================================================
from pyspark.sql import functions as F

print("💎 EXTRACCIÓN DEL CORAZÓN DE PIENZA...")

# 1. Definir el Universo Endógeno (Identificados en ambos lados)
df_endogenos = df_final.filter(
    (F.col("pickup_name") != "unassigned_area") &
    (F.col("dropoff_name") != "unassigned_area")
)

# 2. Métricas de Resumen
total_filas = df_final.count()
total_endogenos = df_endogenos.count()
porcentaje = (total_endogenos / total_filas) * 100

print(f"\n📊 RESUMEN DE CONTROL GEOGRÁFICO:")
print("-" * 50)
print(f"✅ Viajes Totales en Manifold:  {total_filas:,}")
print(f"🔥 Viajes 100% Identificados:   {total_endogenos:,}")
print(f"📈 Tasa de Control Maestro:     {porcentaje:.2f}%")
print("-" * 50)

# 3. Top 15 Rutas Internas (Piedra Rosetta)
print("\n🏆 TOP 15 RUTAS 'SOLO PARA CONOCEDORES' (End-to-End Control):")
df_rutas = df_endogenos.groupBy("pickup_name", "dropoff_name") \
    .count() \
    .withColumn("share_%", (F.col("count") / total_endogenos) * 100) \
    .orderBy(F.col("count").desc())

df_rutas.show(15, truncate=False)

# 4. Análisis de "Short Trips" (Misma zona)
same_zone = df_endogenos.filter(F.col("pickup_name") == F.col("dropoff_name")).count()
print(f"📍 Viajes Intra-Zonales (Mismo Origen/Destino): {same_zone:,} ({ (same_zone/total_endogenos)*100 :.1f}% de los endógenos)")

💎 EXTRACCIÓN DEL CORAZÓN DE PIENZA...

📊 RESUMEN DE CONTROL GEOGRÁFICO:
--------------------------------------------------
✅ Viajes Totales en Manifold:  1,010,001
🔥 Viajes 100% Identificados:   587,051
📈 Tasa de Control Maestro:     58.12%
--------------------------------------------------

🏆 TOP 15 RUTAS 'SOLO PARA CONOCEDORES' (End-to-End Control):


+-----------------------------------+-----------------------------------+-----+------------------+
|pickup_name                        |dropoff_name                       |count|share_%           |
+-----------------------------------+-----------------------------------+-----+------------------+
|tamarindos                         |santa_fe_ibero                     |6218 |1.059192472204289 |
|santa_fe_ibero                     |santa_fe_centro_comercial          |5179 |0.8822061456330028|
|anzures                            |polanco_parque_lincoln             |4903 |0.8351914910288885|
|anzures                            |anahuac_1__bahias__frontera_polanco|4733 |0.8062331892799774|
|anzures                            |terminal_1_aicm                    |4510 |0.7682467111034647|
|anzures                            |rios                               |4388 |0.7474648710248343|
|anahuac_1__bahias__frontera_polanco|terminal_2_aicm                    |4266 |0.7266830309462039|
|santa_fe_

In [ ]:
# ==============================================================================
# CELL 10: THE GEOSPATIAL VALIDATOR (EL FILTRO DE ORO)
# ==============================================================================
# Propósito: Validar la integridad del Manifold contra los polígonos maestros.
# Lógica:   Filtra viajes cuyos nombres semánticos existan en poly.geojson.
# = :===========================================================================
import geopandas as gpd
import json
from pyspark.sql import functions as F

print("🗺️  Cargando Polígonos Maestros desde el sistema local...")

# RUTA SOBERANA EN CODESPACES
PATH_GEOJSON = "/workspaces/pienza/assets/poly.geojson"

try:
    # 1. Leer el GeoJSON (Capa de Verdad Geográfica)
    # Usamos GeoPandas para extraer los nombres oficiales de tus polígonos
    gdf_polys = gpd.read_file(PATH_GEOJSON)

    # Extraemos la lista de nombres únicos definidos en tu mapa
    # Nota: Se asume que la columna es 'name' según tu estándar de Fase 2
    zonas_oficiales = list(set(gdf_polys['name'].unique()))

    print(f"✅ MAPA CARGADO: {len(zonas_oficiales)} zonas semánticas detectadas.")
    print(f"📍 Muestra de zonas oficiales: {zonas_oficiales[:5]}...")

    # 2. APLICAR EL FILTRO DE PUREZA (Spark Mode)
    # Solo permitimos registros donde TANTO el origen como el destino existan en el mapa
    df_pureza = df_final.filter(
        (F.col("pickup_name").isin(zonas_oficiales)) & 
        (F.col("dropoff_name").isin(zonas_oficiales))
    )

    # 3. AUDITORÍA DE PUREZA SEMÁNTICA
    total_manifold = df_final.count()
    total_puros    = df_pureza.count()
    eficiencia     = (total_puros / total_manifold) * 100

    print("\n💎 RESULTADOS DE LA VALIDACIÓN GEOGRÁFICA:")
    print("-" * 55)
    print(f"📦 Registros en Manifold Enriquecido: {total_manifold:,}")
    print(f"✨ Registros 'Mapa-Puros' (Validados): {total_puros:,}")
    print(f"🎯 Eficiencia de la Simulación:       {eficiencia:.2f}%")
    print("-" * 55)

    # 4. EL TOP 10 DEFINITIVO (Certificado por Mapa)
    print("\n🏆 TOP 10 RUTAS CERTIFICADAS POR POLY.GEOJSON:")
    df_pureza.groupBy("pickup_name", "dropoff_name") \
        .count() \
        .orderBy(F.col("count").desc()) \
        .show(10, truncate=False)

    # ESTABLECER EL DATASET FINAL PARA LA SUBIDA A BIGQUERY
    # Este es el dataset que realmente representa el "Expert Rhythm" validado
    df_final_voted = df_pureza
    
    # Cacheamos para la inyección final en la Cell 9
    df_final_voted.cache()

except Exception as e:
    print(f"❌ ERROR CRÍTICO EN LA VALIDACIÓN: {e}")
    print("💡 Tip: Verifica que poly.geojson esté en /workspaces/pienza/assets/")

🗺️  Cargando Polígonos Maestros desde el sistema local...
✅ MAPA CARGADO: 71 zonas semánticas detectadas.
📍 Muestra de zonas oficiales: ['bahias', 'carretera_al_olivo', 'santa_fe_ibero', 'lagos', 'ave_club_de_golf_lomas']...

💎 RESULTADOS DE LA VALIDACIÓN GEOGRÁFICA:
-------------------------------------------------------
📦 Registros en Manifold Enriquecido: 1,010,001
✨ Registros 'Mapa-Puros' (Validados): 185,342
🎯 Eficiencia de la Simulación:       18.35%
-------------------------------------------------------

🏆 TOP 10 RUTAS CERTIFICADAS POR POLY.GEOJSON:
+--------------------+-------------------------+-----+
|pickup_name         |dropoff_name             |count|
+--------------------+-------------------------+-----+
|tamarindos          |santa_fe_ibero           |6218 |
|santa_fe_ibero      |santa_fe_centro_comercial|5179 |
|anzures             |polanco_parque_lincoln   |4903 |
|anzures             |rios                     |4388 |
|santa_fe_ibero      |santa_fe_ibero           |394

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 38192)
Traceback (most recent call last):
  File "/usr/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/home/vscode/.local/lib/python3.11/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/home/vscode/.local/lib/python3.11/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/home/vscode/.local/lib/python3.11/site-packages/pyspark/accumulator